# SCAPI Shopper API — Content assets & content blocks (Page Designer)

Mints a **guest shopper token** via SLAS, then reads storefront content with two
SCAPI **Shopper** APIs:

- **Shopper Content** — `content/{id}` → a library **content asset**
- **Shopper Experience** — `pages/{id}` → a **Page Designer page**, whose
  regions/components are the reusable **content blocks**

Connection settings come from `../dw.json`; the content/page IDs are editable in
the **Parameters** cell. Set them to IDs that exist on *your* site — a fresh
instance may have no content library or Page Designer pages, in which case the
calls return HTTP 404 (the cells handle that gracefully and tell you).

In [ ]:
import json
from pathlib import Path

import httpx

from b2c_tooling_sdk.slas import SlasTokenConfig, get_guest_token

# --- Connection settings (read from dw.json) ---
raw = json.loads(Path("../dw.json").resolve().read_text())
SHORT_CODE = raw["shortCode"]
ORGANIZATION_ID = raw["organizationId"]
SITE_ID = raw["siteId"]
SLAS_CLIENT_ID = raw["slasClientId"]
REDIRECT_URI = raw.get("slasRedirectUri", "http://localhost:3000/callback")

SCAPI_BASE_URL = f"https://{SHORT_CODE}.api.commercecloud.salesforce.com"
print(f"Instance {SHORT_CODE} · org {ORGANIZATION_ID} · site {SITE_ID}")

In [ ]:
# --- Parameters (edit these to IDs that exist on your site) ---
CONTENT_ID = "about-us"   # a library content asset id
PAGE_ID = "homepage"      # a Page Designer page id

In [ ]:
# Mint a guest shopper token (public SLAS client -> PKCE guest flow).
token = await get_guest_token(
    SlasTokenConfig(
        short_code=SHORT_CODE,
        organization_id=ORGANIZATION_ID,
        slas_client_id=SLAS_CLIENT_ID,
        site_id=SITE_ID,
        redirect_uri=REDIRECT_URI,
    )
)
ACCESS_TOKEN = token.access_token
print(f"Guest token minted (customer_id={token.customer_id}, expires_in={token.expires_in}s)")


async def scapi_get(path: str, params: dict | None = None) -> tuple[int, object]:
    """Authenticated GET; returns (status_code, parsed_json_or_text). Never raises on 4xx/5xx."""
    query = {"siteId": SITE_ID, **(params or {})}
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    async with httpx.AsyncClient(timeout=30) as client:
        response = await client.get(f"{SCAPI_BASE_URL}{path}", params=query, headers=headers)
    try:
        return response.status_code, response.json()
    except ValueError:
        return response.status_code, response.text

## 1. Content asset (Shopper Content API)

`GET /content/shopper-content/v1/organizations/{org}/content/{id}?siteId=...`

A content asset's body lives in `c_body` (custom attribute). Set `CONTENT_ID`
above to an asset that exists on your site.

In [ ]:
status, body = await scapi_get(f"/content/shopper-content/v1/organizations/{ORGANIZATION_ID}/content/{CONTENT_ID}")
if status == 200:
    print("id   :", body.get("id"))
    print("name :", body.get("name"))
    text = body.get("c_body") or body.get("body")
    print("body :", (str(text)[:200].strip() + " ...") if text else "(no body attribute)")
elif status == 404:
    print(f"No content asset '{CONTENT_ID}' on site {SITE_ID}. Set CONTENT_ID to a real asset id.")
else:
    print(f"HTTP {status}: {str(body)[:300]}")

## 2. Content blocks — a Page Designer page (Shopper Experience API)

`GET /experience/shopper-experience/v1/organizations/{org}/pages/{id}?siteId=...`

The page is composed of **regions**, each holding **components** — the content
blocks. Set `PAGE_ID` above to a Page Designer page id on your site.

In [ ]:
status, page = await scapi_get(f"/experience/shopper-experience/v1/organizations/{ORGANIZATION_ID}/pages/{PAGE_ID}")
if status == 200:
    print("page :", page.get("id"), "| type:", page.get("typeId"), "| name:", page.get("name"))
    regions = page.get("regions") or []
    print(f"{len(regions)} region(s):")
    for region in regions:
        components = region.get("components") or []
        print(f"  region {region.get('id')!r}: {len(components)} component(s) (content blocks)")
        for component in components:
            print(f"      • {component.get('id')}  [type: {component.get('typeId')}]")
elif status == 404:
    print(f"No Page Designer page '{PAGE_ID}' on site {SITE_ID}. Set PAGE_ID to a real page id.")
else:
    print(f"HTTP {status}: {str(page)[:300]}")